# Proyecto Final: mythos-rag
---

**Curso:** Inteligencia Artificial Generativa con Grandes Modelos del Lenguaje
**Maestria en Informatica** - Escuela Colombiana de Ingenieria Julio Garavito

**Integrantes:**

- Hildebrando Pena Quezada (corpus e indice)
- Lancheros (pipeline rag)  <- completar nombre completo
- Ximena (evaluacion, interfaz y video)  <- completar nombre completo

**Repositorio:** https://github.com/brandoeci/mythos-rag

Un asistente de consulta sobre un corpus de mitologia griega de dominio publico
que responde citando el pasaje exacto, se abstiene cuando la respuesta no esta
en el corpus, y reporta cuando las fuentes se contradicen.

### Problematica
---

Los LLM responden con total seguridad sobre temas especificos, se equivocan, y
nunca dicen de donde sacaron el dato. En mitologia eso es peor, porque no existe
*una* respuesta: quien nace de quien, quien mato a quien, cuantos trabajos
fueron. Hesiodo, Apolodoro y Ovidio no coinciden. Un LLM mezcla las versiones en
una sola respuesta que suena perfecta y no es de nadie, y no hay forma de
verificarla.

(Completar: a quien le sirve, y por que un Ctrl+F no resuelve esto.)

**Objetivo general.** Construir un asistente de consulta sobre un corpus de
mitologia griega de dominio publico que responda en lenguaje natural citando el
pasaje exacto, y que reconozca cuando el corpus no tiene la respuesta o tiene
mas de una version.

**Objetivos especificos.**

1. Armar el corpus e indexarlo: descargar las fuentes, partir en chunks,
   generar embeddings y guardarlas en un vector store con metadata de autor y
   obra.
2. Implementar el pipeline RAG con citacion obligatoria y un guardrail para que
   responda "eso no esta en el corpus" en vez de inventar.
3. Evaluar contra un set de preguntas comparando LLM solo vs. RAG, midiendo
   respuestas correctas, inventadas y abstenciones.

### Trabajos Relacionados
---

(Completar con 1-2 parrafos y referencias. Pistas de donde buscar:)

- El paper original de RAG (Lewis et al., 2020) y por que el retrieval reduce
  alucinacion frente al parametrico puro.
- Trabajo sobre atribucion y respuestas con cita verificable.
- Retrieval hibrido: por que BM25 sigue ganandole al denso en nombres propios.
- Lo que ya existe sobre consulta de corpus literarios o historicos, y en que se
  queda corto para fuentes que se contradicen entre si.

### Metodologia
---

#### Arquitectura

```
corpus (Gutenberg, varias obras)
   |  chunking con metadata {autor, obra, locator}
ChromaDB (denso)  +  BM25 (lexico)
   |
grafo LangGraph:
   retrieve -> grade -> (abstain | detect_conflict -> generate)
   |
Gradio
```

(Completar: justificar cada decision. La tabla de decisiones esta en el README.)

In [ ]:
from corpus import load_corpus, summarize
import pandas as pd

corpus = load_corpus()
pd.DataFrame(summarize(corpus))

#### Indexacion

(Describir chunking, solape, modelo de embeddings multilingue y por que los
embeddings se calculan localmente.)

In [ ]:
from index import chunk_document, build_dense_index, build_lexical_index

chunks = [
    chunk
    for source, text in corpus.items()
    for chunk in chunk_document(text, source.author, source.work)
]
print(f"chunks: {len(chunks)}")

#### El pipeline

(Describir cada nodo del grafo: retrieval hibrido, grading de relevancia, la
arista condicional que lleva a abstencion, la deteccion de conflictos y la
generacion con citas.)

In [ ]:
import rag

rag.STUB = False  # el pipeline real
result = rag.answer("quien fue la madre de heracles?")

print(result.text)
for citation in result.citations:
    print(f"  - {citation}")

### Resultados y Discusion
---

#### Tabla de ablacion

Cada fila agrega una pieza, asi se ve que aporto cada una.

In [ ]:
from evaluate import ablation_table

ablation_table()

#### Analisis

(Completar con los numeros reales. Puntos a cubrir:)

- Cuanto bajo la tasa de invencion al pasar de LLM solo a RAG.
- Si el hibrido mejoro frente al denso, y en que tipo de pregunta. **Si no
  mejoro, decirlo y explicar por que** - eso vale mas que taparlo.
- Cuantos conflictos detecto de los que hay, y los falsos positivos.
- Donde falla: preguntas que requieren juntar varios pasajes, nombres con
  variantes de transliteracion, el costo de que el corpus este en ingles.

No prometemos haber eliminado las alucinaciones. Reportamos que las reducimos y
las hicimos verificables, con los numeros de la tabla.

### Conclusion
---

(Un parrafo.)